In [ ]:
import numpy as np
import trimesh
import matplotlib.pyplot as plt
from scipy.interpolate import splprep, splev
import plotly.graph_objects as go

def calculate_centerline(mesh, ray_count=48, samples=50):
    """
    Calculate the centerline of a mesh by sampling along its Y-axis.
    """
    y_min, y_max = mesh.bounds[:, 1]
    y_vals = np.linspace(y_min, y_max, samples)
    centerline_points = []

    for y_pos in y_vals:
        center_guess = np.array([0.0, y_pos, 0.0])
        _, _, raw_center, _ = get_radial_dimensions(mesh, center=center_guess, ray_count=ray_count)
        if raw_center is not None and len(raw_center) > 0:
            centerline_points.append(np.mean(raw_center, axis=0))

    if len(centerline_points) < 3:
        raise ValueError("Failed to compute centerline: insufficient points.")
    
    return np.array(centerline_points)

def fit_spline_to_centerline(centerline_points, smooth_factor=0.1):
    """
    Fit a spline to smooth the centerline points.
    """
    tck, _ = splprep(centerline_points.T, s=smooth_factor)
    u_new = np.linspace(0, 1, 100)
    smooth_centerline = np.array(splev(u_new, tck)).T
    return smooth_centerline

def visualize_mesh_and_centerline(mesh, centerline_points, smooth_centerline, tip_point, tangent_vector):
    """
    Visualize the mesh, centerline, and a cross-section near the tip.
    """
    # Create a 3D plot
    fig = go.Figure()

    # Add the mesh
    fig.add_trace(go.Mesh3d(
        x=mesh.vertices[:, 0],
        y=mesh.vertices[:, 1],
        z=mesh.vertices[:, 2],
        i=mesh.faces[:, 0],
        j=mesh.faces[:, 1],
        k=mesh.faces[:, 2],
        opacity=0.5,
        color='lightgrey',
        name='Mesh'
    ))

    # Add the raw centerline
    fig.add_trace(go.Scatter3d(
        x=centerline_points[:, 0],
        y=centerline_points[:, 1],
        z=centerline_points[:, 2],
        mode='lines+markers',
        marker=dict(size=4, color='orange'),
        line=dict(color='orange', width=2),
        name='Raw Centerline'
    ))

    # Add the smoothed centerline
    fig.add_trace(go.Scatter3d(
        x=smooth_centerline[:, 0],
        y=smooth_centerline[:, 1],
        z=smooth_centerline[:, 2],
        mode='lines+markers',
        marker=dict(size=4, color='red'),
        line=dict(color='red', width=3),
        name='Smoothed Centerline'
    ))

    # Add the tip point
    fig.add_trace(go.Scatter3d(
        x=[tip_point[0]],
        y=[tip_point[1]],
        z=[tip_point[2]],
        mode='markers',
        marker=dict(size=8, color='blue'),
        name='Tip Point'
    ))

    # Add the tangent vector at the tip
    fig.add_trace(go.Cone(
        x=[tip_point[0]],
        y=[tip_point[1]],
        z=[tip_point[2]],
        u=[tangent_vector[0]],
        v=[tangent_vector[1]],
        w=[tangent_vector[2]],
        sizemode='absolute',
        sizeref=0.2,
        anchor='tail',
        colorscale=[[0, 'blue'], [1, 'blue']],
        name='Tangent Vector'
    ))

    # Update layout
    fig.update_layout(
        title="Mesh with Centerline and Tip Visualization",
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z",
            aspectmode="data"
        ),
        margin=dict(l=0, r=0, t=40, b=0)
    )

    # Show the plot
    fig.show()

def get_radial_dimensions(mesh, center, ray_count=48):
    """
    Compute radial dimensions of the mesh at a given center point.
    """
    angles = np.linspace(0, 2 * np.pi, ray_count, endpoint=False)
    inner, outer = [], []

    for angle in angles:
        direction = np.array([np.cos(angle), 0, np.sin(angle)])
        points, _, _ = mesh.ray.intersects_location([center], [direction])
        if len(points) >= 2:
            distances = np.linalg.norm(points - center, axis=1)
            sorted_indices = np.argsort(distances)
            inner.append(points[sorted_indices[0]])
            outer.append(points[sorted_indices[-1]])

    if not inner:
        return None, None, None, None

    inner = np.array(inner)
    outer = np.array(outer)
    raw_center = (inner + outer) / 2
    avg_minor_radius = np.mean(np.linalg.norm(outer - inner, axis=1)) / 2

    return inner, outer, raw_center, avg_minor_radius

# Main function to load, process, and visualize a single mesh
def main():
    # Path to the mesh file
    mesh_path = "Meshes/OBJ/Ac_DA_1_2.obj"  # Replace with your mesh file path

    # Load the mesh
    mesh = trimesh.load_mesh(mesh_path)
    if isinstance(mesh, trimesh.Scene):
        mesh = mesh.dump(concatenate=True)

    # Calculate the centerline
    centerline_points = calculate_centerline(mesh)

    # Fit a spline to smooth the centerline
    smooth_centerline = fit_spline_to_centerline(centerline_points)

    # Select a point near the tip
    tip_index = int(len(smooth_centerline) * 0.95)  # Adjust 0.95 as needed
    tip_point = smooth_centerline[tip_index]

    # Compute the tangent vector at the tip
    tangent_vector = smooth_centerline[tip_index + 1] - smooth_centerline[tip_index - 1]
    tangent_vector /= np.linalg.norm(tangent_vector)  # Normalize

    # Visualize the mesh, centerline, and tip
    visualize_mesh_and_centerline(mesh, centerline_points, smooth_centerline, tip_point, tangent_vector)

if __name__ == "__main__":
    main()